In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
from matplotlib.ticker import FuncFormatter, LogLocator,ScalarFormatter
from matplotlib.ticker import MultipleLocator
import matplotlib.lines as mlines
from scipy.interpolate import interp1d
%run constants_functions.ipynb

bar = 1e5
sec_per_year = 60*60*24*365
n_orb=n*1
t_eval = np.arange(0, 4.6, 0.001)*1e9 # years
t_save = t_eval[::10]
Venuscolor = 'darkorange'
altVenuscolor = 'skyblue'
TLcolor = 'silver'
noteqcolor = '#1b9e77'

def solar_evolution(time):
    L_sun = 3.828e26 # W
    to = 4.57e9
    t_years = time*1
    Lum = L_sun*(1+(2/5)*(1-t_years/to))**(-1)
    S = Lum/(4*np.pi*a**2)
    return(S)

def critical_rotation(S, S_RG, sign):
    T_guess = S*1
    for i in range(len(T_guess)):
        T_guess[i] = 500 # days
        sig = sign*2*np.pi/(T_guess[i]*60*60*24) - n
        albedo = interp_albedo([abs(sig), S[i]]) #+ 0.01
        ASR = S[i]*(1-albedo)/4 
        while ASR < S_RG and T_guess[i] > 1:
            sig = sign*2*np.pi/(T_guess[i]*60*60*24) - n
            albedo = interp_albedo([abs(sig), S[i]]) #+ 0.01
            ASR = S[i]*(1-albedo)/4
            T_guess[i] -= 1
    return(T_guess)

n_Yang = np.pi*2/(365*60*60*24)
rotation_periods = np.array([1,16,64,128,256,365])
sigs_Yang = 2*np.pi/(rotation_periods[::-1]*60*60*24)-n_Yang
all_solar_constants = np.array([1379.4943747591249, 1414.104535777566, 1466.9267604350466, 1509.9349498795164, 1558.9332174259953, 1611.1126981332104, 1806.115794878488, 2004.5074574305263, 2106.921933698868, 2393.6433608367, 2695.81674655818, 2993.4288665912286, 3282.220865044703, 3579.9226244639185])
from scipy.interpolate import RegularGridInterpolator
albedo_matrix = np.load('../model/input_data/albedo_Yang_matrix.npy')
interp_albedo = RegularGridInterpolator((sigs_Yang, all_solar_constants), albedo_matrix,bounds_error=False, fill_value=0) 

critical_rot_pro = critical_rotation(solar_evolution(t_save), 300, 1)
critical_rot_ret = -critical_rotation(solar_evolution(t_save), 300, -1)

plt.rcParams.update({
    # FIGURE & AXES SIZING
    "figure.figsize": (6, 4),  # Set figure size (width, height) in inches
    "figure.dpi": 300,  # High resolution for publication quality
    "axes.titlesize": 14,  # Title font size
    "axes.labelsize": 7,  # Axis label font size
    "axes.labelpad": 3,  # Padding for axis labels

    # TICKS & GRIDLINES
    "xtick.labelsize": 15,  # X-axis tick label font size
    "ytick.labelsize": 15,  # Y-axis tick label font size
    "xtick.major.size": 10,  # Major tick size
    "ytick.major.size": 10,
    "xtick.minor.size": 5,  # Minor tick size
    "ytick.minor.size": 5,
    "xtick.direction": "in",  # Ticks inside the plot
    "ytick.direction": "in",

    # GRID SETTINGS
    "axes.grid": True,  # Enable grid by default
    "grid.color": "gray",  # Grid line color
    "grid.linestyle": "--",  # Dashed grid lines for major ticks
    "grid.linewidth": 0.6,  # Thin grid lines
    "grid.alpha": 0.5,  # Slight transparency for better readability

    # MINOR GRID SETTINGS
    "grid.linestyle": ":",  # Dotted style for minor grid lines
    "grid.linewidth": 0.4,  # Thinner minor grid lines

    # LINES & MARKERS
    "lines.linewidth": 2,  # Line thickness
    "lines.markersize": 6,  # Marker size

    # FONT SETTINGS
    "font.family": "sans-serif",  # Use serif font (e.g., Times New Roman)
    "font.size": 12,  # General font size

    # LEGEND SETTINGS
    "legend.frameon": False,  # No legend border
    "legend.fontsize": 10,
    "legend.loc": "best",

    # SAVEFIG SETTINGS
    "savefig.dpi": 300,  # High resolution for saving figures
    "savefig.transparent": True,  # Transparent background for easy overlay in publications
    "savefig.bbox": "tight",  # Prevents cutting off labels when saving

    # AXES SPINES
    "axes.spines.top": True,  # Show top spine
    "axes.spines.right": True,  # Show right spine
    "axes.spines.left": True,
    "axes.spines.bottom": True,
})

In [ ]:
## from scipy.optimize import fsolve

from scipy.integrate import solve_ivp
from scipy.integrate import quad
from scipy.special import sph_harm
from scipy.special import gamma
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import time 
import logging 

import sys
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import MultipleLocator

# Get the parameter value passed as a command-line argument

rheo_model = 'Andrade'
kappa_sw = 0.54e-4 # atmospheric shortwave absorption coefficient
kappa_lw = 0.1e-4 # atmospheric shortwave absorption coefficient
alph = 0.2
tau_e = 500*60*60*24*365



In [ ]:
G = 6.67e-11 # gravitational constant
AU = 1.49e11 # [m], astronomical unit
a = 0.723*AU # [m], Venus' semi-major axis
R_V = 6.0518e6 # [m], radius of Venus
M_V = 4.8685e24 # [kg], mass of Venus
M_sun = 1.989e30 # [kg], mass of Sun
Ifactor = 0.336 # moment of inertia factor for Venus
C = Ifactor*M_V*R_V**2 # moment of inertia Venus on spin axis
deltEd = 1.3e-5 # departure from dynamical ellipticity, CL2003 pg. 3

## for gravitational tide ##

k2 = 0.295 # 2nd order love number 
n = 2*np.pi/(224.701*60*60*24)# mean motion, 2pi/T Venus

sec_per_year = 60*60*24*365

Kg = (G*M_sun**2*R_V**5)/(C*a**6) # gravitational tide constant


## for CMF ##

Rc = 3.2e6 # m, suggested core radius of Venus, Correia 2003
Cc = 0.084*C # moment of inertia of core
deltE_c = 4*deltEd # core dynamical ellipticity anomaly 

gamma_el = 0.75 # C2003, I, pg. 8, eq. 50
ed_ec = 1/4 # non-hydrostatic, 4/3 for hydrostatic 


## for thermal tide

g = 8.87 # m/s2
sigma_SB = 5.67e-8 # Stefan-Boltzmann constant [W m-2 K-4]


## constants used in Salazar & Wordsworth, 2024
Cs_land = 1000000 # heat capacity of surface, [J kg-1]
cp = 1000 # specific heat of air, [J kg-1 K-1]
rho_mean = 5204 # density of planet, [kg m-3]
D = 1.66 # diffusivity constant 
R = 188 # specific gas constant, [J kg-1 K-1]
Cd = 0.0034 # surface drag coefficient
chi = 0.17 # reference thermal coupling factor
bar = 1e5

## lumiosity vs time
L_sun = 3.828e26 # W, luminosity of Sun
to = 4.6e9 # yr, age of solar system

Ka = (3*M_sun*R_V**3)/(5*C*rho_mean*a**3) # atmospheric thermal tide torque constant 

# Modern Venus atmosphere parameters

Omega_Venus = -2*np.pi/(244*60*60*24) # [rad/s], rotation rate
alpha_Venus = 0.77 # albeo
ps_Venus = 92e5 # [Pa] surface pressure
S_Venus = 2624.3 # [Wm-2], solar constant

# for Andrade rheology
ke = 0.25 # elastic love number
kf = 0.928 # fluid love number
tau_tot = kf*tau_e/ke

## load ocean tidal dissipation from Green 2019:

Q_data = np.load('../model/input_data/Q_withIT_330.npy')
sig_data = np.load('../model/input_data/sig_330_Green.npy')
f_interp_Qocean = interp1d(sig_data, Q_data, kind = 'linear',fill_value="extrapolate")

## albedo parameterization 
def alpha_fit(S, Omega):
    sigma = Omega-n
    sigma_crit = 0.744e-6* (S/1366)**(3/4)
    alpha_slow = 0.79*(1-np.exp(-(0.86*S/1366)))+0.08 
    alpha_rapid = 0.34
    alpha = alpha_rapid + (alpha_slow-alpha_rapid)/np.sqrt(1+(sigma/sigma_crit)**2) # based on dT 
    return(alpha)


def Tbar(S, alpha, tau_sw, tau_lw):
    k = tau_sw/tau_lw 
    F_bar = S*(1-alpha)*np.exp(-tau_sw)/np.pi # average incident stellar radiation at surface (Wm-2)
    SLW = S*(1-alpha)/8*(1+D/k - (1+D/k)*np.exp(-k*tau_lw)) # surface downwelling longwave (Wm-2), Equation (15)
    T_bar = np.power((F_bar+SLW)/sigma_SB, 1/4) # average surface temperature (K)
    return(T_bar)

def wind_speed(S, alpha, tau_sw, tau_lw, ps): # Us [m/s], Equation (23)
    T_bar = Tbar(S, alpha, tau_sw, tau_lw)
    T_eq = (S*(1-alpha)/(4*sigma_SB))**(1/4) # equilibrium surface temperature (K)
    Us = np.power(R/Cd * np.maximum((T_bar - T_eq),0) * (S/2)*(1-alpha)*np.exp(-tau_sw)*(1-np.exp(-tau_lw))/ps,1/3) 
    return(Us)

def qo_wo(S, alpha, tau_sw, tau_lw, ps, outgassing=False): # output q_o (defined in Equation 29) and w_o
    k = tau_sw/tau_lw
    F_bar = S*(1-alpha)*np.exp(-tau_sw)/np.pi # average incident stellar radiation at surface (Wm-2)
    SLW = S*(1-alpha)/8*(1+D/k - (1+D/k)*np.exp(-k*tau_lw)) # surface downwelling longwave (Wm-2), Equation (15)
    T_bar = np.power((F_bar+SLW)/sigma_SB, 1/4) # average surface temperature (K)
    delta_F = 1/6 * np.sqrt(15/(2*np.pi)) # sph. harm decomp coefficient of insolation field
    T_eq = (S*(1-alpha)/(4*sigma_SB))**(1/4) # equilibrium surface temperature (K)
    Us = wind_speed(S, alpha, tau_sw, tau_lw, ps)
    circ_strength = Us/Uso # circulation strength, Equation (22)
    delt_p = chi*ps*circ_strength # Equation (23)
    Cs = cp*delt_p/g + Cs_land # heat capacity of surface, including atmosphere
    if outgassing:
        w_o = 3.77e-7 # from Leconte 2015
    else:
        w_o = 4*sigma_SB*T_bar**3/Cs # thermal equilibrium frequency
    qo = -(1/4)*delta_F*S*(1-alpha)*np.exp(-tau_sw)*delt_p/(F_bar + SLW)*np.sqrt(10/(3*np.pi))
    return(qo, w_o)

def torque_analytic_forcingfreq(S, alpha, tau_sw, tau_lw, ps, Omega, n, m, l, outgassing=False): # q_tilde [Pa], Equation (29)
    qo, w_o = qo_wo(S, alpha, tau_sw, tau_lw, ps, outgassing)
    sigma = (m*Omega-l*n)
    torque = -qo*sigma/w_o/(1+(sigma/w_o)**2) # Equation (29)
    return(torque)


def thermal_tide_torque(M, R, a,S, alpha, tau_sw, tau_lw, ps, Omega_list, n, m, l, outgassing=False): # T_a, Equation (32)
    Ka = -3*M*R**3/((5*rho_mean*a**3))
    q_tilde = torque_analytic_forcingfreq(S, alpha, tau_sw, tau_lw, ps, Omega_list, n, m,l,outgassing)
    return(-Ka*q_tilde)

def gravitational_tide_torque(M, R, a,Q,Qn, k2, Omega_list, n, rheo_model='Constant-Q'): # T_g, Equation (4)
    Kg = -G*M**2*R**5/a**6
    if rheo_model == 'Constant-Q':
        bg = k2/Q * np.sign(Omega_list-n) 
    elif rheo_model == 'CL2003':
        bg = k2/Q * np.sign(Omega_list-n)  * (1-(1-Q/Qn)**(np.absolute(2*(Omega_list-n))/n))
    elif rheo_model == 'Andrade':
        bg = Andrade(2*(Omega_list-n))
    else:
        print('Error: please enter valid rheology model')
        bg = np.nan
    return(Kg*bg)

def Andrade(sig):
    B_sig = 1 + np.absolute(sig*tau_tot)**(1-alph) * (tau_e/tau_tot)**(1-alph)*np.sin(alph*np.pi/2)*gamma(1+alph)
    A_sig = (sig*tau_tot)*(1 + np.maximum(np.absolute(sig*tau_tot),1e-5)**(-alph) * (tau_e/tau_tot)**(1-alph)*np.cos(alph*np.pi/2)*gamma(1+alph) )
    k2Q = (kf-ke) * B_sig*sig*tau_tot/(A_sig**2 + B_sig**2) 
    return(k2Q)

def critical_rotation(S, S_RG, sign):
    T_guess = S*1
    for i in range(len(T_guess)):
        T_guess[i] = 500 # days
        sig = sign*2*np.pi/(T_guess[i]*60*60*24) - n
        albedo = interp_albedo([abs(sig), S[i]]) #+ 0.01
        ASR = S[i]*(1-albedo)/4 
        while ASR < S_RG:
            sig = sign*2*np.pi/(T_guess[i]*60*60*24) - n
            albedo = interp_albedo([abs(sig), S[i]]) #+ 0.01
            ASR = S[i]*(1-albedo)/4
            T_guess[i] -= 1
    return(T_guess)

#critical_rot_pro = critical_rotation(solar_evolution(t_save), 300, 1)
critical_rot_ret = -critical_rotation(solar_evolution(t_save), 300, -1)


Uso = wind_speed(1137, 0.2, 0.00001, 1, 1*bar) # U_so [m/s]    


def modern_Venus_tuning(x, rheo_model, Q, Qn):
    A = x
    rot_eq = gravitational_tide_torque(M_sun, R_V, a,Q,Qn, k2, Omega_Venus, n, rheo_model) + thermal_tide_torque(M_sun, R_V, a,S_Venus, alpha_Venus, 0.54+A*(1+np.tanh((ps_Venus/bar - 50)/15))*0.5, 1 * (ps_Venus/bar)**(0.8), ps_Venus, Omega_Venus, n,2,2,True)
    return(rot_eq)
A_sw = float(fsolve(modern_Venus_tuning, [0.5], args = ('Andrade', np.nan, np.nan)))
tau_lw_Venus =  1 * (ps_Venus/bar)**(0.8)
tau_sw_Venus = 0.54+A_sw*(1+np.tanh((ps_Venus/bar - 50)/15))*0.5



def spin_evolution_steam(t,x, rheo_model, tau_sw_steam, integrate=True):
    L = x[0] # L = C*omega
    obl = x[1] # obliquity [rad]
    omega = L/C 
    X = L*np.cos(obl)
    Lum = L_sun*(1+(2/5)*(1-t/to))**(-1) # Gough, 1981, solar evolution in time
    S = Lum/(4*np.pi*a**2) # solar constant, Wm-2

    pad_omega = np.sign(omega)*max(abs(omega), 1e-7) # for stable passage through omega = 0
        
    #########################################
    ########## Gravitational Tide ###########
    #########################################

    if rheo_model == 'Ocean_Andrade':
        def b_g_sig(sigma):
            k2Q_ocean = 0.2/f_interp_Qocean(sigma) * np.tanh((sigma)/(0.1*n)) * 0.5*(1+np.tanh((hab-0.15)/0.05)) # smooth change of sign and hab
            k2_Andrade = Andrade(sigma)
            return(k2Q_ocean + k2_Andrade)

    elif rheo_model == 'Andrade':
        def b_g_sig(sigma):
            return(Andrade(sigma))
    elif rheo_model == 'Constant-Q':
        def b_g_sig(sigma):
            if sigma >= 0:
                sign = 1
            else: 
                sign = -1
            return(k2*sign/Q * (1-(1-Q/Qn)**(abs(sigma)/n)))

    
        
    domegadt_grav = -Kg*(b_g_sig(omega)*(3/4 * X**2/L**2 * (1-X**2/L**2)) + b_g_sig(omega-2*n)*(3/16 * (1+X/L)**2 * (1-X**2/L**2))+ \
                b_g_sig(omega+2*n)*(3/16 * (1-X/L)**2 * (1-X**2/L**2) ) + b_g_sig(2*omega)*(3/8 * (1-X**2/L**2)**2) + \
                b_g_sig(2*omega-2*n)*(3/32 * (1+X/L)**4) + b_g_sig(2*omega+2*n)*(3/32 * (1-X/L)**4))

    
    
    dobldt_grav = -Kg  * np.sin(obl)/pad_omega * (b_g_sig(2*n)*(9/16 * np.sin(obl)**2) + \
                   b_g_sig(omega)*3/4 * np.cos(obl)**3 - b_g_sig(omega-2*n)*3/16*(1+np.cos(obl))**2 * (2-np.cos(obl)) +\
                   b_g_sig(omega+2*n)*3/16*(1-np.cos(obl))**2 * (2+np.cos(obl)) + b_g_sig(2*omega)*3/8*np.sin(obl)**2*np.cos(obl) +\
                   -b_g_sig(2*omega-2*n)*3/32*(1+np.cos(obl))**3 +  b_g_sig(2*omega+2*n)*3/32*(1-np.cos(obl))**3)
    
        
    ###########################################
    ########## Core Mantle Friction ###########
    ###########################################

    Ed = kf*R_V**5/(3*G*C) * pad_omega**2 + deltEd # elipticity factor 
    Ec = Ed/ed_ec # core elipticity factor
    alpha = 3*n**2/(2*pad_omega) * Ed # precession constant (rad/s)

    E = visc/(abs(pad_omega)*Rc**2) # Ekman number 
    
    kappa = 2.62*Cc*abs(omega)*np.sqrt(E) # viscous friction, CL003 Eq. 53
    
    Kf = kappa/(gamma_el*C) * (n/pad_omega)**4 *(3/2 * ed_ec)**2 # CL2003, Eq. 72, CMF torque constant

    domegadt_CMF = -omega*Kf*np.cos(obl)**2*np.sin(obl)**2 # CL2003, Eq. 71 
    dobldt_CMF = -Kf*np.cos(obl)**3*np.sin(obl) # CL2003, Eq. 74

    domegadt = domegadt_grav  + domegadt_CMF 
    dobldt = dobldt_grav   + dobldt_CMF 
    if integrate:
        return(C*domegadt*sec_per_year, dobldt*sec_per_year)
    else:
        return(C*domegadt_grav, C*0, C*domegadt_CMF)

def spin_evolution_hab(t,x, rheo_model,tau_lw_habitable, tau_sw_habitable, p_s_habitable, delta_t_s, Q, Qn, integrate=True):
    L = x[0] # L = C*omega
    obl = x[1] # obliquity [rad]
    omega = L/C 
    X = L*np.cos(obl)
    Lum = L_sun*(1+(2/5)*(1-t/to))**(-1) # Gough, 1981, solar evolution in time
    S = Lum/(4*np.pi*a**2) # solar constant, Wm-2

    pad_omega = np.sign(omega)*np.maximum(np.abs(omega), 1e-7) # for stable passage through omega = 0
    
    ## check if habitability is possible 
    #albedo = alpha_fit(S, pad_omega*np.sign(np.cos(obl))) # get albedo from cloud-rotation feedback
    sig_albedo = np.abs(pad_omega*np.sign(np.cos(obl)) - n) # |sigma|, assume symmetric in LOD
    albedo = float(interp_albedo([sig_albedo, S])) # interpolate Yang 2014 50-m data
    S_avg = S*(1-albedo)/4 # ASR
    ps = p_s_habitable
    tau_lw = tau_lw_habitable
    tau_sw = tau_sw_habitable
    albedo_in = albedo*1

    #########################################
    ########## Gravitational Tide ###########
    #########################################

    if rheo_model == 'Ocean_Andrade':
        def b_g_sig(sigma):
            k2Q_ocean = 0.2/f_interp_Qocean(sigma) * np.tanh((sigma)/(0.1*n)) # smooth change of sign and hab
            k2_Andrade = Andrade(sigma)
            return(k2Q_ocean + k2_Andrade)

    elif rheo_model == 'Andrade':
        def b_g_sig(sigma):
            return(Andrade(sigma))
    elif rheo_model == 'Constant-Q':
        def b_g_sig(sigma):
            if sigma >= 0:
                sign = 1
            else: 
                sign = -1
            return(k2*sign/Q * (1-(1-Q/Qn)**(abs(sigma)/n)))

    
        
    domegadt_grav = -Kg*(b_g_sig(omega)*(3/4 * X**2/L**2 * (1-X**2/L**2)) + b_g_sig(omega-2*n)*(3/16 * (1+X/L)**2 * (1-X**2/L**2))+ \
                b_g_sig(omega+2*n)*(3/16 * (1-X/L)**2 * (1-X**2/L**2) ) + b_g_sig(2*omega)*(3/8 * (1-X**2/L**2)**2) + \
                b_g_sig(2*omega-2*n)*(3/32 * (1+X/L)**4) + b_g_sig(2*omega+2*n)*(3/32 * (1-X/L)**4))

    
    
    dobldt_grav = -Kg  * np.sin(obl)/pad_omega * (b_g_sig(2*n)*(9/16 * np.sin(obl)**2) + \
                   b_g_sig(omega)*3/4 * np.cos(obl)**3 - b_g_sig(omega-2*n)*3/16*(1+np.cos(obl))**2 * (2-np.cos(obl)) +\
                   b_g_sig(omega+2*n)*3/16*(1-np.cos(obl))**2 * (2+np.cos(obl)) + b_g_sig(2*omega)*3/8*np.sin(obl)**2*np.cos(obl) +\
                   -b_g_sig(2*omega-2*n)*3/32*(1+np.cos(obl))**3 +  b_g_sig(2*omega+2*n)*3/32*(1-np.cos(obl))**3)
    
        
    ###########################################
    ########## Core Mantle Friction ###########
    ###########################################

    Ed = kf*R_V**5/(3*G*C) * pad_omega**2 + deltEd # elipticity factor 
    Ec = Ed/ed_ec # core elipticity factor
    alpha = 3*n**2/(2*pad_omega) * Ed # precession constant (rad/s)

    E = visc/(abs(pad_omega)*Rc**2) # Ekman number 
    
    kappa = 2.62*Cc*abs(omega)*np.sqrt(E) # viscous friction, CL003 Eq. 53
    
    Kf = kappa/(gamma_el*C) * (n/pad_omega)**4 *(3/2 * ed_ec)**2 # CL2003, Eq. 72, CMF torque constant

    domegadt_CMF = -omega*Kf*np.cos(obl)**2*np.sin(obl)**2 # CL2003, Eq. 71 
    dobldt_CMF = -Kf*np.cos(obl)**3*np.sin(obl) # CL2003, Eq. 74
    
    
###########################
###### Thermal Tide #######
###########################

    ## ExoTides (Salazar & Wordsworth, 2024)

    
    def b_a_sig(omega, m, l):
        q_tilde = torque_analytic_forcingfreq(S, albedo_in, tau_sw, tau_lw, ps, omega, n, m, l)
        return(-q_tilde)
    


    
    domegadt_atm = -Ka*(b_a_sig(omega,1,0)*(3/4 * X**2/L**2 * (1-X**2/L**2)) + b_a_sig(omega,1,2)*(3/16 * (1+X/L)**2 * (1-X**2/L**2))+ \
                b_a_sig(omega,1,-2)*(3/16 * (1-X/L)**2 * (1-X**2/L**2) ) + b_a_sig(omega,2,0)*(3/8 * (1-X**2/L**2)**2) + \
                b_a_sig(omega,2,2)*(3/32 * (1+X/L)**4) + b_a_sig(omega,2,-2)*(3/32 * (1-X/L)**4))
    

    
    dobldt_atm = -Ka  * np.sin(obl)/pad_omega * (b_a_sig(omega, 0, -2)*(9/16 * np.sin(obl)**2) + \
                    b_a_sig(omega,1,0)*3/4 * np.cos(obl)**3 - b_a_sig(omega,1,2)*3/16*(1+np.cos(obl))**2 * (2-np.cos(obl)) +\
                    b_a_sig(omega,1,-2)*3/16*(1-np.cos(obl))**2 * (2+np.cos(obl)) + b_a_sig(omega,2,0)*3/8*np.sin(obl)**2*np.cos(obl) +\
                    -b_a_sig(omega,2,2)*3/32*(1+np.cos(obl))**3 +  b_a_sig(omega,2,-2)*3/32*(1-np.cos(obl))**3)

    ## sum torques together ## 

    domegadt = domegadt_grav  + domegadt_CMF + domegadt_atm
    dobldt = dobldt_grav   + dobldt_CMF + dobldt_atm
    if integrate:
        return(C*domegadt*sec_per_year, dobldt*sec_per_year)
    else:
        return(C*domegadt_grav, C*domegadt_atm, C*domegadt_CMF)

def spinout(t, x, rheo_model,tau_lw_habitable, tau_sw_habitable, p_s_habitable, delta_t_s, Q, Qn):
        L, obl  = x
        omega = (L/C)
        pad_omega = np.sign(omega)*max(abs(omega), 1e-7)
        Lum = L_sun*(1+(2/5)*(1-t/to))**(-1)
        S = Lum/(4*np.pi*a**2)
        sig_albedo = abs(pad_omega*np.sign(np.cos(obl)) - n) # |sigma|, assume symmetric in LOD
        albedo = float(interp_albedo([sig_albedo, S])) # interpolate Yang 2014 50-m data
        #albedo = alpha_fit(S, pad_omega*np.sign(np.cos(obl)))
        S_avg = S*(1-albedo)/4
        return(S_avg - S_RG)
spinout.terminal = True   # stop integration at first crossing
spinout.direction = 1.0   # prefer upward crossing 


def spin_evolution_Venus(t,x, rheo_model,p_s_habitable, t_end_hab, Q, Qn,Q_ocean, ramp, integrate=True):
    L = x[0] # L = C*omega
    obl = x[1] # obliquity [rad]
    omega = L/C 
    X = L*np.cos(obl)
    Lum = L_sun*(1+(2/5)*(1-t/to))**(-1) # Gough, 1981, solar evolution in time
    S = Lum/(4*np.pi*a**2) # solar constant, Wm-2

    pad_omega = np.sign(omega)*np.maximum(np.abs(omega), 1e-7) # for stable passage through omega = 0

    ps = (ps_Venus-p_s_habitable) * ((t-t_end_hab)/(to-t_end_hab))**ramp + p_s_habitable # outgassing rate set with ramp 
    tau_lw = 1 * (ps/bar)**(0.8)
    tau_sw = 0.54+A_sw*(1+np.tanh((ps/bar - 50)/15))*0.5
    albedo_in = alpha_Venus
            
    #########################################
    ########## Gravitational Tide ###########
    #########################################

    if rheo_model in ['Andrade','Ocean_Andrade']:
        def b_g_sig(sigma):
            return(Andrade(sigma))
    elif rheo_model == 'Constant-Q':
        def b_g_sig(sigma):
            if sigma >= 0:
                sign = 1
            else: 
                sign = -1
            return(k2*sign/Q * (1-(1-Q/Qn)**(abs(sigma)/n)))

    
        
    domegadt_grav = -Kg*(b_g_sig(omega)*(3/4 * X**2/L**2 * (1-X**2/L**2)) + b_g_sig(omega-2*n)*(3/16 * (1+X/L)**2 * (1-X**2/L**2))+ \
                b_g_sig(omega+2*n)*(3/16 * (1-X/L)**2 * (1-X**2/L**2) ) + b_g_sig(2*omega)*(3/8 * (1-X**2/L**2)**2) + \
                b_g_sig(2*omega-2*n)*(3/32 * (1+X/L)**4) + b_g_sig(2*omega+2*n)*(3/32 * (1-X/L)**4))

    
    
    dobldt_grav = -Kg  * np.sin(obl)/pad_omega * (b_g_sig(2*n)*(9/16 * np.sin(obl)**2) + \
                   b_g_sig(omega)*3/4 * np.cos(obl)**3 - b_g_sig(omega-2*n)*3/16*(1+np.cos(obl))**2 * (2-np.cos(obl)) +\
                   b_g_sig(omega+2*n)*3/16*(1-np.cos(obl))**2 * (2+np.cos(obl)) + b_g_sig(2*omega)*3/8*np.sin(obl)**2*np.cos(obl) +\
                   -b_g_sig(2*omega-2*n)*3/32*(1+np.cos(obl))**3 +  b_g_sig(2*omega+2*n)*3/32*(1-np.cos(obl))**3)
    
        
    ###########################################
    ########## Core Mantle Friction ###########
    ###########################################

    Ed = kf*R_V**5/(3*G*C) * pad_omega**2 + deltEd # elipticity factor 
    Ec = Ed/ed_ec # core elipticity factor
    alpha = 3*n**2/(2*pad_omega) * Ed # precession constant (rad/s)

    E = visc/(abs(pad_omega)*Rc**2) # Ekman number 
    
    kappa = 2.62*Cc*abs(omega)*np.sqrt(E) # viscous friction, CL003 Eq. 53
    
    Kf = kappa/(gamma_el*C) * (n/pad_omega)**4 *(3/2 * ed_ec)**2 # CL2003, Eq. 72, CMF torque constant

    domegadt_CMF = -omega*Kf*np.cos(obl)**2*np.sin(obl)**2 # CL2003, Eq. 71 
    dobldt_CMF = -Kf*np.cos(obl)**3*np.sin(obl) # CL2003, Eq. 74
    
    
###########################
###### Thermal Tide #######
###########################

    ## ExoTides (Salazar & Wordsworth, 2024)

    
    def b_a_sig(omega, m, l):
        q_tilde = torque_analytic_forcingfreq(S, albedo_in, tau_sw, tau_lw, ps, omega, n, m, l,True)
        return(-q_tilde)
    


    
    domegadt_atm = -Ka*(b_a_sig(omega,1,0)*(3/4 * X**2/L**2 * (1-X**2/L**2)) + b_a_sig(omega,1,2)*(3/16 * (1+X/L)**2 * (1-X**2/L**2))+ \
                b_a_sig(omega,1,-2)*(3/16 * (1-X/L)**2 * (1-X**2/L**2) ) + b_a_sig(omega,2,0)*(3/8 * (1-X**2/L**2)**2) + \
                b_a_sig(omega,2,2)*(3/32 * (1+X/L)**4) + b_a_sig(omega,2,-2)*(3/32 * (1-X/L)**4))
    

    
    dobldt_atm = -Ka  * np.sin(obl)/pad_omega * (b_a_sig(omega, 0, -2)*(9/16 * np.sin(obl)**2) + \
                    b_a_sig(omega,1,0)*3/4 * np.cos(obl)**3 - b_a_sig(omega,1,2)*3/16*(1+np.cos(obl))**2 * (2-np.cos(obl)) +\
                    b_a_sig(omega,1,-2)*3/16*(1-np.cos(obl))**2 * (2+np.cos(obl)) + b_a_sig(omega,2,0)*3/8*np.sin(obl)**2*np.cos(obl) +\
                    -b_a_sig(omega,2,2)*3/32*(1+np.cos(obl))**3 +  b_a_sig(omega,2,-2)*3/32*(1-np.cos(obl))**3)

    ## sum torques together ## 

    domegadt = domegadt_grav  + domegadt_CMF + domegadt_atm
    dobldt = dobldt_grav   + dobldt_CMF + dobldt_atm
    if integrate:
        return(C*domegadt*sec_per_year, dobldt*sec_per_year)
    else:
        return(C*domegadt_grav, C*domegadt_atm, C*domegadt_CMF)
    


x_end = np.log10(4.6e9)
t_eval = 10**np.linspace(5, x_end, 10000) # years
t_resurface = 4e9 # resurfacing event 
def run_Venus_evolution(delta_t_s, L_o, obl_o, tau_lw, tau_sw, ps, Q, Qn,Q_ocean,tau_lw_Venus, tau_sw_Venus,tau_sw_steam, ramp_rate): 
    ## steam state
    t_steam_end = delta_t_s             # when steam ends (already in seconds)
    t_resurface = 4.0e9                 # resurfacing time
    # t_hab_end will be determined by your event; initialize to t_resurface for now

    # Subset for steam/hab BEFORE you know t_hab_end:
    mask_steam = (t_eval <= t_steam_end)
    t_eval_steam = t_eval[mask_steam]

    mask_hab_pre = (t_eval > t_steam_end) & (t_eval <= t_resurface)
    t_eval_hab = t_eval[mask_hab_pre]

    sol_steam = solve_ivp(spin_evolution_steam, [0,delta_t_s],[L_o, obl_o], method='BDF',max_step=1e8,t_eval=t_eval_steam,args = (rheo_model, tau_sw_steam))
    obl_steam = 180/np.pi * sol_steam.y[1] # rad --> degrees
    omega_steam = sol_steam.y[0]/C
    hab_steam = np.zeros(len(obl_steam)) 
    pst_steam = np.zeros(len(obl_steam)) + 10e5
    
    ## hab state
    Lum_endsteam = L_sun*(1+(2/5)*(1-delta_t_s/to))**(-1)
    S_endsteam = Lum_endsteam/(4*np.pi*a**2)
    sig_albedo = abs(omega_steam[-1]*np.sign(np.cos(np.radians(obl_steam[-1]))) - n) # |sigma|, assume symmetric in LOD
    albedo_endsteam = interp_albedo([sig_albedo, S_endsteam]) # interpolate Yang 2014 50-m data
    #albedo_endsteam = alpha_fit(S_endsteam, omega_steam[-1]*np.sign(np.cos(np.radians(obl_steam[-1]))))

    S_avg_endsteam = S_endsteam*(1-albedo_endsteam)/4
    
    if S_avg_endsteam >= S_RG: # if already not hab
        t_hab_end = delta_t_s
        obl_hab = [] # rad --> degrees
        omega_hab = []
        hab_hab = []
        pst_hab = []
        initial_rot = sol_steam.y[0][-1]/C
        initial_obl = 180/np.pi * sol_steam.y[1][-1]
    else:
        sol_hab = solve_ivp(spin_evolution_hab, [t_eval_hab[0],t_resurface],[C*float(omega_steam[-1]), np.radians(float(obl_steam[-1]))], method='BDF',max_step=1e8,t_eval=t_eval_hab,args = (rheo_model,tau_lw, tau_sw, ps, delta_t_s, Q, Qn), events=[spinout])
        t_hab_end = float(sol_hab.t_events[0][0]) if sol_hab.t_events[0].size > 0 else t_resurface
        obl_hab = 180/np.pi * sol_hab.y[1] # rad --> degrees
        omega_hab = sol_hab.y[0]/C
        initial_rot = omega_hab[-1]
        initial_obl = obl_hab[-1]
        hab_hab = np.zeros(len(obl_hab)) + 1
        pst_hab = np.zeros(len(obl_hab)) + ps
    mask_steam = (t_eval <= t_steam_end)
    mask_hab   = (t_eval > t_steam_end) & (t_eval <= t_hab_end)
    mask_venus = (t_eval > t_hab_end)
    
    mask_steam = (t_eval <= t_steam_end)
    mask_hab   = (t_eval > t_steam_end) & (t_eval <= t_hab_end)
    mask_venus = (t_eval > t_hab_end)

    # Subsets to pass to the last phase
    t_eval_Venus = t_eval[mask_venus]
    ## venus state 
    y0_Venus = np.array([float(C*initial_rot), float(np.radians(initial_obl))], dtype=float)

    sol_Venus = solve_ivp(spin_evolution_Venus, [t_eval_Venus[0],t_eval_Venus[-1]],y0_Venus, method='BDF',max_step=1e8,t_eval=t_eval_Venus,args = (rheo_model,ps,t_hab_end, Q, Qn,Q_ocean, ramp_rate))
    obl_Venus = 180/np.pi * sol_Venus.y[1] # rad --> degrees
    omega_Venus = sol_Venus.y[0]/C
    hab_Venus = np.zeros(len(obl_Venus))-1
    pst_Venus = (ps_Venus-ps) * ((t_eval_Venus-t_hab_end)/(4.6e9-t_hab_end))**ramp_rate + ps

    eq_rotation_rate = np.hstack([omega_steam,omega_hab, omega_Venus])
    eq_obl = np.hstack([obl_steam,obl_hab, obl_Venus]) 
    habitable = np.hstack([hab_steam,hab_hab, hab_Venus])
    ps_t = np.hstack([pst_steam,pst_hab, pst_Venus])

    return(eq_rotation_rate, eq_obl, habitable, ps_t)

In [ ]:
## grab sample solution

all_solutions = pd.read_pickle('../baseline/all_solutions.pkl')
Venus_all = all_solutions.loc[all_solutions['final state']=='Venus']
Venus_everhab = Venus_all.loc[Venus_all['ever hab?']==1]

In [ ]:
## for clarity, get a long-lived habitable state with short steam state
sample_solution = Venus_everhab.loc[((Venus_everhab['t_end']-Venus_everhab['t_start']) > 0.2) & ((Venus_everhab['$t_s$']) < 2e6)].iloc[2]

In [ ]:
sample_solution

In [ ]:
## sample solution spin-out
S_RG = 300
ts = sample_solution['$t_s$']
To = sample_solution['$T_o$']
t_end = sample_solution['t_end']*1e9
epso = sample_solution['$\\epsilon_o$']
ramp_rate = 0.001

sol = run_Venus_evolution(ts, C*2*np.pi/(To*60*60*24), np.radians(epso), 1, 0.54, 2*bar, 100, 100,100,tau_lw_Venus, tau_sw_Venus,10, ramp_rate)

In [ ]:
omega_sol = np.array(sol)[0]
obl_sol = np.array(sol)[1]
h_sol = np.array(sol)[2]
ps_sol = np.array(sol)[3]

In [ ]:
omega_grav = t_eval*0
omega_atm = t_eval*0
omega_CMF = t_eval*0
for i in range(len(t_eval)):
    t = t_eval[i]
    if t <= ts:
        omega_grav[i], omega_atm[i], omega_CMF[i] = spin_evolution_steam(t, [omega_sol[i]*C, np.radians(obl_sol[i])],'Andrade',10,False)
    elif ts < t <= t_end:
        omega_grav[i], omega_atm[i], omega_CMF[i] = spin_evolution_hab(t, [omega_sol[i]*C, np.radians(obl_sol[i])],'Andrade',1, 0.54, 2*bar,ts,100,100,False)
    else:
        omega_grav[i], omega_atm[i], omega_CMF[i] = spin_evolution_Venus(t, [omega_sol[i]*C, np.radians(obl_sol[i])],'Andrade',2*bar, t_end, 100, 100,100,ramp_rate, False)

In [ ]:
np.save('omega_grav.npy', omega_grav)
np.save('omega_atm.npy', omega_atm)
np.save('omega_CMF.npy', omega_CMF)
np.save('sample_spinout.npy',sol)

In [ ]:
#t_eval = np.linspace(0, 4.6, 100000)*1e9 # years
sol = np.load('sample_spinout.npy')
Omega_t, obl_t, h_t, ps_t = sol
domegadt_grav = np.load('omega_grav.npy')
domegadt_atm = np.load('omega_atm.npy')
domegadt_CMF = np.load('omega_CMF.npy')
n_orb=n*1
#critical_rot_ret = -critical_rotation(solar_evolution(t_eval), 300, -1)


In [ ]:

import matplotlib.gridspec as gridspec

from matplotlib.ticker import FuncFormatter, MultipleLocator,ScalarFormatter

fig = plt.figure(figsize=(10, 8))
gs = gridspec.GridSpec(2, 1, height_ratios=[1, 1], hspace=0.05, wspace=0)

# --- Top main plot (occupies full row) ---
ax = fig.add_subplot(gs[0, :])  # span all 3 columns in first row

spinout = Venus_all.loc[(Venus_all['t_end']-Venus_all['t_start'] > 0.4) & (Venus_all['t_end']-Venus_all['t_start'] < 0.6)]

period_ticks = np.array([400,100,10,1])
omega_ticks = 2*np.pi/(period_ticks*60*60*24)

which = spinout

color=Venuscolor
ts = 1912521.529314
where_hab = h_t*1
ax.fill_between(t_save, -critical_rot_ret, 0,color='red',alpha=0.15,linestyle='--',linewidth=1)
# major ticks every 10
ax.xaxis.set_major_locator(MultipleLocator(1))

ax.plot(t_eval, 2*np.pi/(Omega_t)/60/60/24, '--', color=color, linewidth=3)
ax.plot(t_eval[where_hab>0.3],2*np.pi/(Omega_t[where_hab>0.3])/60/60/24, '-', color=color, linewidth=4)


ax.set_ylim(400,0.9)
ax.set_yscale('log')#, linthresh=5e-7)
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.yaxis.get_major_formatter().set_scientific(False)
ax.set_yticks(period_ticks)
#ax.yaxis.set_major_formatter(FuncFormatter(rate_to_period))
#ax.minorticks_off()


ax2 = ax.twinx()
# ax2.plot(t_eval,obl_t, '--', color='grey', linewidth=3)
# ax2.plot(t_eval[where_hab>0],obl_t[where_hab>0.3],'-', color='grey', linewidth=4)
ax2.set_ylim(0,181)
#print(np.vstack(which['$\Omega(t)$'])[rand_int][where_hab>0.3][-1])
ax2.set_yticks([0,30,60,90,120,150,180])
ax2.set_ylabel('Obliquity (deg.)', fontsize=12,color='grey')
ax2.spines['right'].set_color('k')
ax2.tick_params(axis='y', colors='k')
#ax[i,j].annotate(str(np.round(ps,1)) + ' bar', (3, -5), fontsize=15)
ax.grid(alpha=0.3)
ax.set_ylabel('Rotation Period (days)', color=Venuscolor,fontsize=14)
ax.spines['left'].set_color(Venuscolor)
ax.tick_params(axis='y', which='both',colors=Venuscolor)
ax.set_xlabel('Time (yr)', fontsize=15)

ax2.spines['right'].set_color('grey')
ax2.tick_params(axis='y', colors='grey')

ax.xaxis.set_ticks_position('top')      # move ticks
ax.xaxis.set_label_position('top')  
# minor ticks at every point
ax.xaxis.set_minor_locator(MultipleLocator(0.5))

ax.grid(True, which = 'both',linewidth=0.5)

ax.set_xscale('log')
ax.set_xlim(1e5, 4.6e9)

# ax.axvline(x = ts, color='k', linestyle='-')
# ax.axvline(x = t_eval[where_hab>0.3][-1], color='k', linestyle='-')

ax.annotate('Runaway Greenhouse', xy=(1.2e5,1.9), color='red', fontsize=15)
sec_per_year = 60*60*24*365


# --- Bottom plot: dOmega/dt (entire timespan) ---
ax_domega = fig.add_subplot(gs[1])

# Plot all components over full time range
ax_domega.plot(t_eval, 1e-16 * domegadt_grav , color='brown', linewidth=2, label='Grav. Tide')
ax_domega.plot(t_eval, 1e-16 * domegadt_atm , color='blue', linewidth=2, label='Atm. Tide')
ax_domega.plot(t_eval, 1e-16 * domegadt_CMF , color='green', linewidth=2, label='CMF')
ax_domega.plot(t_eval, 1e-16 * (domegadt_grav + domegadt_atm + domegadt_CMF) , 'k-', linewidth=4,label='Net')

ax_domega.set_ylabel(r' Torque $(\times 10^{16}$ N$\cdot$m)', fontsize=14)
ax_domega.set_xlabel("Time (yr)", fontsize=15)
ax_domega.set_xscale('log')
ax_domega.set_xlim(1e5, 4.6e9)
ax_domega.set_ylim(-5, 5)
ax.grid(True, which='both',linewidth=0.5, alpha=1)
ax_domega.grid(True, which='both',linewidth=0.5, alpha=1)
ax_domega.legend(ncol=4, loc='lower center', bbox_to_anchor=(0.5, 0.04), fontsize=15)

#plt.savefig('Figure_3.png', bbox_inches='tight')
